# Task 2: Visualizing fire scars through false color

Author: Veronica Moran

https://github.com/veronica-moran/eds220-hwk4.git

## Purpose

This notebook uses Landsat 8 satellite imagery and fire perimeter data to visualize the Palisades and Eaton fire scars. These fires occured in Los Angeles County in January 2025. False Color imagery is used to highlight burned areas against surrounding vegetation.

## Highlights

- Clean the fire perimeter data to make it easier to work with, filtering data and creating a new dataframe with only the most recent Palisades and Eaton fire perimeters.
- Cleaning Landsat data by restoring spatial information, adjusting scale to get true color image, and setting null values to 0
- Reprojecting the fire perimeter data to match CRS of Landsat data
- Creating a false color map using shortwave infrared, near-infrared, and red bands on the Landsat data and overlaying fire perimeters.

## About the data

Simplified collection of bands (red, green, blue, near-infrared and shortwave infrared) from the Landsat Collection 2 Level-2 atmospherically corrected surface reflectance data, collected by the Landsat 8 satellite.

The data was retrieved from the [Microsoft Planetary Computer data catalog](https://planetarycomputer.microsoft.com/dataset/landsat-c2-l2) and clipped to an area surrounding the fire perimeters. This data should be used for visualization and educational purposes only and can be accessed as landsat8-2025-02-23-palisades-eaton.nc in the course shared data [Google Drive](https://drive.google.com/drive/u/1/folders/1USqhiMLyN8GE05B8WJmHabviJGnmAsLP).

The second dataset contains fire perimeters for the Eaton and Palisades fires. The data was retrieved from the [AVIRIS Data Portal](https://popo.jpl.nasa.gov/mmgis-aviris/?mission=LA_Fires). Most recent fire perimeter was from January 12, 2025. The data was accessed November 15, 2026.

## References

[1] Microsoft Planetary Computer. Landsat Collection 2 Level-2. https://planetarycomputer.microsoft.com/dataset/landsat-c2-l2

[2] NASA Jet Propulsion Laboratory. AVIRIS data portal: LA fires. Retrieved November 15, 2026. https://popo.jpl.nasa.gov/mmgis-aviris/?mission=LA_Fires

[3] Riebeek, H. (2014, March 14). Why is that forest red and that cloud blue? How to interpret a false-color satellite image. NASA Earth Observatory. https://science.nasa.gov/earth/earth-observatory/how-to-interpret-a-false-color-satellite-image/

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import xarray as xr
import rioxarray
import matplotlib.patheffects as pe
from shapely.geometry import box
pd.set_option('display.max_columns', None)

In [ ]:
# Read fire perimeters
fp = os.path.join('data', 'fire_perimeters.geojson')
perimeters = gpd.read_file(fp)

# Read Landsat imagery
fp = os.path.join('data', 'landsat8-2025-02-23-palisades-eaton.nc')
bands = xr.open_dataset(fp)


# Fire Perimeter Data Exploration

In [ ]:
perimeters.head(10)

In [ ]:
# Checking column names, non-null count and dtype
perimeters.info()

In [ ]:
# Checking if the CRS is geographic
print(perimeters.crs.is_geographic)

# Checking if the CRS is projected
print(perimeters.crs.is_projected)

# Checking CRS
perimeters.crs

In [ ]:
# Checking unique fire names
perimeters['name'].unique()

The fire perimeter dataset contains 34 rows and 9 columns, with no missing values. The data include fire names, acreage, perimeter area and length, data sources, observation dates, and fire perimeter geometries. There are nine fire names which includes different capitalizations for Palisades and Eaton fires and one unknown fire name. Acres dtype is an object, instead of numeric. The CRS is WGS 84 (EPSG:4326), a geographic reference system.

In [ ]:
# Plotting the fire perimeters
perimeters.plot()

# Perimeters Data Cleaning

In [ ]:
# Updating name columns to lowercase fire names 
perimeters['name'] = perimeters['name'].str.lower()

In [ ]:
# Updating columns to lowercase and relace double underscore with single underscore
perimeters.columns = perimeters.columns.str.lower().str.replace('__', '_')

In [ ]:
# Filtering to palisades and eaton fires,sorting by newest date
perimeters = perimeters[
    perimeters['name'].isin(['palisades', 'eaton'])
].sort_values('datetime', ascending = False)

# Grouping by fire name, creating new dataframe with newest perimeter for each fire
palisades_eaton = perimeters.groupby('name').head(1).reset_index(drop=True)

palisades_eaton

In [ ]:
# Plotting fire perimeters

palisades_eaton.plot()

# NetCDF Data Exploration

In [ ]:
# Checking bands information
bands

The Landsat dataset contains five spectral bands: red, green, blue, near-infrared (nir08), and shortwave infrared (swir22). Each band contains 3,888,156 values stored as float32. Spatial dimensions are y (1,418 rows) and x (2,742 columns). The x and y coordinates units are in meters, with a spatial resolution of 30 meters. The CRS is WGS 84 / UTM zone 11N (EPSG:32611), which is a projected coordinate reference system.

# Restoring Spatial Information

In [ ]:
# Checking bands CRS
print(bands.rio.crs)

In [ ]:
# Printing CRS
print(bands.spatial_ref.crs_wkt)

In [ ]:
# Recovering geospatial information
bands = bands.rio.write_crs(bands.spatial_ref.crs_wkt)

print(bands.rio.crs)

# True Color Image

In [ ]:
# Converting data to dataarray() and plotting
bands[['red', 'green', 'blue']].to_dataarray().plot.imshow(robust = True)

In [ ]:
# Checking null value count
bands.isnull().sum()

In [ ]:
# Replacing null values with 0
bands = bands.fillna(0)

In [ ]:
bands.isnull().sum()

In [ ]:
# Plotting true color image red, green and blue bands
bands[['red', 'green', 'blue']].to_dataarray().plot.imshow(robust = True)

In [ ]:
# Plotting false color image, shortwave infrared, near-infrared, red
falsecolor = bands[['swir22', 'nir08', 'red']].to_dataarray().plot.imshow(robust = True)
falsecolor

In [ ]:
# Reprojecting perimeter data to match bands data CRS
palisades_eaton = palisades_eaton.to_crs(bands.rio.crs)
assert palisades_eaton.crs == bands.rio.crs

In [ ]:
# Plotting raster and perimeters
fig, ax = plt.subplots(figsize=(10, 8))
ax.axis('off')

# Plotting raster
bands[['swir22', 'nir08', 'red']].to_dataarray().plot.imshow(robust = True)

# Plotting Perimeter
palisades_eaton.boundary.plot(ax=ax,
                edgecolor='blue',
                linewidth=1,
                label = 'Fire Perimeter')

# Adding title
ax.set_title('Palisades and Eaton Fire Scars\nLos Angeles County 2025')

# Adding legenf
ax.legend(loc='lower left')


# Adding fire names as text annotations
for x, y, name in zip(palisades_eaton.geometry.centroid.x,
                      palisades_eaton.geometry.centroid.y,
                      palisades_eaton['name']):
    ax.text(x, y + 4500, name.title() + ' Fire',
            fontsize=10,
            color='white',
            fontweight='bold',
            ha='right',
           path_effects=[
               pe.withStroke(linewidth=2,
                             foreground='black')
            ])
    
# Adding data sources
ax.text(0, -0.04,
        'Imagery: Landsat 8, Microsoft Planetary Computer (February 23, 2025)\n'
        'Fire perimeters: NASA/JPL AVIRIS Data Portal (January 12, 2025, pre-containment)',
        transform=ax.transAxes,
        fontsize=8,
        va='top')

plt.show()

Palisades and Eaton fire scars, outlined in blue, are shown using false-color Landsat 8 imagery. Shortwave infrared (SWIR), near-infrared (NIR), and red bands are displayed in the red, green and blue channels to highlight differences in land cover. SWIR is strongly absorbed by water and moisture in leaves. When vegetation is burned or removed, exposed soil and burned areas reflect more SWIR and less NIR. Healthy vegetation strongly reflects NIR and due to chlorophyll strongly absorbs red light. These differences in how land cover absorbs or reflects light cause burn scars to appear red or rust colored, healthy vegetation to appear bright green, and water to appear black or dark blue[3].